### Libraries

In [29]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from collections import Counter
from tqdm import tqdm

In [2]:
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import classification_report, accuracy_score

In [4]:
import evaluate

### Setup

In [30]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


Choose between 2 different sets of data with different ratios

In [3]:
# Load the datasets (Ratio of 70:15:15 for train, validation, and test sets)
train_df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/modeldata/train70.csv")
val_df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/modeldata/val15.csv")
test_df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/modeldata/test15.csv")

In [ ]:
# Load the datasets (Ratio of 80:10:10 for train, validation, and test sets)
train_df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/modeldata/train80.csv")
val_df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/modeldata/val10.csv")
test_df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/modeldata/test10.csv")

In [4]:
print(train_df['label'].value_counts(normalize=True))
print(val_df['label'].value_counts(normalize=True))
print(test_df['label'].value_counts(normalize=True))

label
1    0.50025
0    0.49975
Name: proportion, dtype: float64
label
0    0.505536
1    0.494464
Name: proportion, dtype: float64
label
1    0.504371
0    0.495629
Name: proportion, dtype: float64


In [31]:
X_train = train_df['verse'].values
X_val = val_df['verse'].values
X_test = test_df['verse'].values

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

In [45]:
X_train = X_train.tolist()
X_valid = X_val.tolist()
X_test = X_test.tolist()

y_train = y_train.tolist()
y_valid = y_val.tolist()
y_test = y_test.tolist()

In [5]:
# Fixes the [WinError 3] by explicitly setting a local cache directory
cache_dir = "C:/Users/User/Documents/devanasokan_fyp/huggingface_cache"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

# Tell Hugging Face to use this directory
os.environ['HF_HOME'] = cache_dir

# This turns off the annoying symlink warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [32]:
from collections import Counter

counter = Counter()

for sentence in X_train:
    counter.update(sentence.split())

vocab = {
    "<PAD>":0,
    "<UNK>":1
}

for word, freq in counter.items():
    if freq >= 2:
        vocab[word] = len(vocab)

In [34]:
def encode(sentence):

    return [
        vocab.get(word,1)
        for word in sentence.split()
    ]

In [35]:
MAX_LEN = 200

def pad(sequence):

    if len(sequence) > MAX_LEN:
        return sequence[:MAX_LEN]

    return sequence + [0]*(MAX_LEN-len(sequence))

In [49]:
class LyricsDataset(Dataset):

    def __init__(self,texts,labels):

        self.texts=texts
        self.labels=labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,index):

        x=torch.tensor(
            pad(
                encode(
                    self.texts[index]
                )
            ),
            dtype=torch.long
        )

        y=torch.tensor(
            self.labels[index],
            dtype=torch.float
        )

        return x,y

In [50]:
BATCH_SIZE = 32

train_dataset = LyricsDataset(X_train,y_train)
valid_dataset = LyricsDataset(X_val,y_val)
test_dataset = LyricsDataset(X_test,y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)

### Tokenization

### Model Training

In [51]:
class RNNClassifier(nn.Module):

    def __init__(self,vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            128,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=128,
            hidden_size=64,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            64,
            1
        )

    def forward(self,x):

        x = self.embedding(x)

        output,hidden = self.rnn(x)

        hidden = hidden.squeeze(0)

        hidden = self.dropout(hidden)

        return self.fc(hidden)

In [52]:
model = RNNClassifier(
    len(vocab)
).to(device)

In [53]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [54]:
def train(model, dataloader, criterion, optimizer, device):

    model.train()

    running_loss = 0
    predictions = []
    labels = []

    for inputs, targets in tqdm(dataloader):

        inputs = inputs.to(device)
        targets = targets.to(device).float().unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, targets)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        preds = (torch.sigmoid(outputs) >= 0.5).int()

        predictions.extend(preds.cpu().numpy())

        labels.extend(targets.cpu().numpy())

    epoch_loss = running_loss / len(dataloader)

    epoch_acc = accuracy_score(labels, predictions)

    return epoch_loss, epoch_acc

In [55]:
def evaluate(model, dataloader, criterion, device):

    model.eval()

    running_loss = 0
    predictions = []
    labels = []

    with torch.no_grad():

        for inputs, targets in tqdm(dataloader):

            inputs = inputs.to(device)
            targets = targets.to(device).float().unsqueeze(1)

            outputs = model(inputs)

            loss = criterion(outputs, targets)

            running_loss += loss.item()

            preds = (torch.sigmoid(outputs) >= 0.5).int()

            predictions.extend(preds.cpu().numpy())

            labels.extend(targets.cpu().numpy())

    epoch_loss = running_loss / len(dataloader)

    epoch_acc = accuracy_score(labels, predictions)

    return epoch_loss, epoch_acc

In [56]:
NUM_EPOCHS = 20
PATIENCE = 3

best_val_loss = float("inf")
patience_counter = 0

train_losses = []
valid_losses = []

train_accs = []
valid_accs = []

In [57]:
for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    train_loss, train_acc = train(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = evaluate(
        model,
        valid_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    valid_losses.append(val_loss)

    train_accs.append(train_acc)
    valid_accs.append(val_acc)

    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Train Loss : {train_loss:.4f}")

    print(f"Val Acc  : {val_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        patience_counter = 0

        torch.save(
            model.state_dict(),
            "best_rnn_model.pt"
        )

        print("Best model saved.")

    else:

        patience_counter += 1

        print(f"EarlyStopping Counter: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:

            print("Early stopping.")

            break


Epoch 1/20


100%|██████████| 108/108 [00:00<00:00, 626.32it/s]


Train Acc  : 0.5080
Train Loss : 0.6922
Val Acc  : 0.5198
Val Loss   : 0.6907
Best model saved.

Epoch 2/20


100%|██████████| 108/108 [00:00<00:00, 577.19it/s]


Train Acc  : 0.5149
Train Loss : 0.6900
Val Acc  : 0.5160
Val Loss   : 0.6907
EarlyStopping Counter: 1/3

Epoch 3/20


100%|██████████| 108/108 [00:00<00:00, 587.53it/s]


Train Acc  : 0.5162
Train Loss : 0.6899
Val Acc  : 0.5157
Val Loss   : 0.6921
EarlyStopping Counter: 2/3

Epoch 4/20


100%|██████████| 108/108 [00:00<00:00, 602.52it/s]

Train Acc  : 0.4961
Train Loss : 0.6979
Val Acc  : 0.5070
Val Loss   : 0.6916
EarlyStopping Counter: 3/3
Early stopping.


In [58]:
model.load_state_dict(
    torch.load("best_rnn_model.pt")
)

<All keys matched successfully>

### Model Evaluation

In [ ]:
import accelerate
print(accelerate.__version__)

1.13.0


In [ ]:
import accelerate
import transformers
print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")

Accelerate version: 1.13.0
Transformers version: 5.3.0


In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
